# M7 · Express behavioral promises as tests

Tests compare actual behavior with independently understood expectations. Direct calls first make assertions observable; a small temporary file then demonstrates real pytest collection, parametrization, and fixtures.

**Start:** Before class, from `teaching-snippets/`, run `uv sync --extra dev` and select its Python environment as the notebook kernel. Notebook-editor/kernel setup is separate classroom preparation. Restart the kernel and run top to bottom. All teaching Python is in the editable cells below; no repository imports or external lesson loaders are needed.

**Explore:** Predict → run → observe → change one input. Expected results assume the supplied values and a fresh run; changing earlier state may change later results. To return to the baseline, restart and rerun from the start.

**Actual checks:** The labeled pytest cells run a subprocess using the selected kernel’s interpreter and visible temporary source files. They print real test output and exit status when you execute them. No repository tests or production code are changed. Deliberate failures are handled or captured so the supplied sequence can continue to its recovery.

## Exploration 1 — build and observe the mechanism

### 1. State the behavior to be checked

An operation makes a promise about inputs and outcomes. Tests need that promise before choosing assertions.

**Before running:** Predict the result or state change.

In [ ]:
def calculate_line_total(price_paise, quantity):
    if type(quantity) is not int or quantity <= 0:
        raise ValueError("Quantity must be a positive integer")
    return price_paise * quantity

print(calculate_line_total(49900, 2))


**Change and explain:** State one accepted and one rejected quantity.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Two cables total 99800 paise.

</details>

### 2. Compare with an independent expectation

An assertion checks a meaningful property. Compute the expected answer from the requirement, not by calling the implementation again.

**Before running:** Predict the result or state change.

In [ ]:
actual = calculate_line_total(49900, 2)
assert actual == 99800
print("Checked two cables:", actual)


**Change and explain:** Change the expected amount deliberately and read the resulting failure; restore it afterward.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The exact amount assertion holds for the correct calculation.

</details>

### 3. Define a test and actually call it

A test function organizes arrange, act, and assert. Definition alone does not execute its checks.

**Before running:** Predict the result or state change.

In [ ]:
def test_two_cables():
    price_paise = 49900
    quantity = 2
    actual = calculate_line_total(price_paise, quantity)
    assert actual == 99800
    return actual

print("Test result:", test_two_cables())


**Change and explain:** Explain which line executes the assertion.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Calling the function exercises its assertion; pytest discovery is demonstrated later.

</details>

### 4. Observe a green but weak assertion

Wrong behavior can preserve the property a weak test checks. An integer result alone does not establish multiplication.

**Before running:** Predict the result or state change.

In [ ]:
def broken_line_total(price_paise, quantity):
    return price_paise + quantity

actual = broken_line_total(49900, 2)
assert isinstance(actual, int)
print("Type check passed for:", actual)


**Change and explain:** Which independently expected amount would expose the defect?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The type assertion passes for the wrong amount 49902.

</details>

### 5. Expose the defect and recover

A useful comparison distinguishes correct behavior. The catch here is only for a controlled teaching failure.

**Before running:** Predict the result or state change.

In [ ]:
try:
    assert actual == 99800
except AssertionError:
    print("Expected 99800; received", actual)

repaired = calculate_line_total(49900, 2)
assert repaired == 99800
print("Repaired result:", repaired)


**Change and explain:** Why must an ordinary failing test not catch and suppress its own assertion?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The strong comparison detects the defect; the correct operation returns 99800.

</details>

### 6. Check accepted and rejected boundaries

A rejection is an observable outcome. Require the specific exception rather than any unrelated failure.

**Before running:** Predict the result or state change.

In [ ]:
import pytest
assert calculate_line_total(0, 1) == 0
for quantity in [0, -1, True]:
    with pytest.raises(ValueError, match="positive integer"):
        calculate_line_total(49900, quantity)
    print("Checked rejection:", repr(quantity))


**Change and explain:** Add a floating-point quantity; distinguish zero price from zero quantity.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

A free product is accepted; invalid quantities raise the expected error.

</details>

## Exploration 2 — change a condition and inspect the consequence

### 7. Express one rule with several cases

Parametrization records separate cases for pytest. Explicit direct calls here let you explore the assertion immediately.

**Before running:** Predict the result or state change.

In [ ]:
@pytest.mark.parametrize("quantity, expected", [(1, 49900), (2, 99800)])
def test_line_total(quantity, expected):
    assert calculate_line_total(49900, quantity) == expected

for quantity, expected in [(1, 49900), (2, 99800)]:
    test_line_total(quantity, expected)
    print("Checked:", quantity, expected)


**Change and explain:** These calls are manual exploration, not pytest collection. The file-based run below exercises collection.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Both known amounts satisfy the same assertion structure.

</details>

### 8. Use fresh owners for state checks

Stateful tests need known starting conditions. Independent instances expose shared-state defects.

**Before running:** Predict the result or state change.

In [ ]:
class ShoppingCart:
    def __init__(self):
        self.quantities = {}
    def add(self, product_id, quantity):
        self.quantities[product_id] = quantity
    def checkout(self):
        snapshot = tuple(self.quantities.items())
        self.quantities.clear()
        return snapshot

cart = ShoppingCart()
other = ShoppingCart()
cart.add("1", 2)
assert other.quantities == {}
print(cart.quantities, other.quantities)


**Change and explain:** Change one cart; predict the other before checking.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Only the chosen instance changes.

</details>

### 9. Retain a regression about snapshots

A test for a historical promise must examine the record after later state changes.

**Before running:** Predict the result or state change.

In [ ]:
def test_snapshot_survives_later_changes():
    cart = ShoppingCart()
    cart.add("1", 2)
    snapshot = cart.checkout()
    assert cart.quantities == {}
    cart.add("1", 5)
    assert snapshot == (("1", 2),)
    return snapshot, cart.quantities

print(test_snapshot_survives_later_changes())


**Change and explain:** What defect would a live reference to the cart introduce?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The snapshot has two cables while the current cart has five.

</details>

### 10. Put a small suite where pytest can discover it

Pytest discovers files, not notebook globals. This visible standalone suite repeats the familiar operation and supplies a fresh fixture per case.

**Before running:** Predict the result or state change.

In [ ]:
test_source = """
import pytest

def line_total(price_paise, quantity):
    return price_paise * quantity

@pytest.fixture
def cart():
    return {}

@pytest.mark.parametrize("quantity, expected", [(1, 49900), (2, 99800)])
def test_total(quantity, expected, cart):
    assert cart == {}
    cart["1"] = quantity
    assert line_total(49900, cart["1"]) == expected
"""
print(test_source)


**Change and explain:** Edit the visible suite to add a third independently understood expected amount.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

This cell displays the actual source that the following cells write and run.

</details>

### 11. Create a throwaway test file

Temporary resources keep the experiment separate from the learner’s project. No repository test files are edited.

**Before running:** Predict the result or state change.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
directory = TemporaryDirectory()
test_path = Path(directory.name) / "test_shop.py"
test_path.write_text(test_source, encoding="utf-8")
print(test_path.name, test_path.exists())


**Change and explain:** Identify how the test filename and function names permit discovery.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The temporary test file contains the visible suite.

</details>

### 12. Run discovery and read individual outcomes

A real pytest invocation collects the parametrized cases and resolves the fixture. Its exit status and report provide evidence.

**Before running:** Predict the result or state change.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_shop.py", "-q", "-p", "no:cacheprovider"],
    cwd=directory.name, capture_output=True, text=True, check=False,
)
print(result.stdout)
print(result.stderr)
print("Exit status:", result.returncode)
directory.cleanup()


**Change and explain:** After editing the suite above, rerun its write and run cells; inspect each reported case.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The supplied correct suite is expected to report two passed cases and exit status zero. This is predicted, not pre-captured output.

</details>

## Apply the mechanism in Nano-Zon

Follow the [M7 lab guide](../../projects/nano-zon/labs/m07/README.md). The codealong isolates the mechanism; the lab combines it with the supplied application.

State the principle, point to the code that expresses it, then explain one limit of the observed result.

### Apply the checks to project files

The [M7 source guide](../../teaching-snippets/m07/README.md) supplies the formal test/runner commands. The notebook is a small exploration; the Nano-Zon lab checks the complete application.

**Authoring status:** This notebook contains editable teaching source, not captured execution output. Syntax and structure can be inspected independently of execution; runtime delivery review remains pending.